[Kinetica](https://www.kinetica.com/) is a database with integrated support for vector similarity search.

It supports:

- exact and approximate nearest neighbor search
- L2 distance, inner product, and cosine distance

This notebook shows how to use the Kinetica vector store (`Kinetica`).

This needs an instance of Kinetica which can easily be setup using the instructions given here - [installation instruction](https://www.kinetica.com/developer-edition/).

In [ ]:
pip install -qU langchain-kinetica

We want to use `OpenAIEmbeddings` so we have to get the OpenAI API Key.

In [1]:
import getpass
import os

from langchain_openai import OpenAIEmbeddings

if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API Key:")

embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

You must set the database connection in the following environment variables. If you are using a virtual environment you can set them in the `.env` file of the project:

* `KINETICA_URL`: Database connection URL (e.g. `http://localhost:9191`)
* `KINETICA_USER`: Database user
* `KINETICA_PASSWD`: Secure password.

In [2]:
# Kinetica needs the connection to the database.
# Set these environment variables:
from gpudb import GPUdb

from langchain_kinetica import KineticaSettings, KineticaVectorstore

kdbc = GPUdb.get_connection()
k_config = KineticaSettings(kdbc=kdbc)
k_config

2026-02-05 09:33:45.777 INFO     [GPUdb] Connected to Kinetica! (host=http://localhost:19191 api=7.2.3.4 server=7.2.3.5)


KineticaSettings(kdbc=<gpudb.gpudb.GPUdb object at 0x121fb7a10>, database='langchain', table='langchain_kinetica_embeddings', metric='l2')

In [3]:
from uuid import uuid4

from langchain_core.documents import Document

document_1 = Document(
    page_content="I had chocolate chip pancakes and scrambled eggs for"
    " breakfast this morning.",
    metadata={"source": "tweet"},
)

document_2 = Document(
    page_content="The weather forecast for tomorrow is cloudy and overcast"
    ", with a high of 62 degrees.",
    metadata={"source": "news"},
)

document_3 = Document(
    page_content="Building an exciting new project with LangChain - come check it out!",
    metadata={"source": "tweet"},
)

document_4 = Document(
    page_content="Robbers broke into the city bank and stole $1 million in cash.",
    metadata={"source": "news"},
)

document_5 = Document(
    page_content="Wow! That was an amazing movie. I can't wait to see it again.",
    metadata={"source": "tweet"},
)

document_6 = Document(
    page_content="Is the new iPhone worth the price? Read this review to find out.",
    metadata={"source": "website"},
)

document_7 = Document(
    page_content="The top 10 soccer players in the world right now.",
    metadata={"source": "website"},
)

document_8 = Document(
    page_content="LangGraph is the best framework for building stateful"
    ", agentic applications!",
    metadata={"source": "tweet"},
)

document_9 = Document(
    page_content="The stock market is down 500 points today due to"
    " fears of a recession.",
    metadata={"source": "news"},
)

document_10 = Document(
    page_content="I have a bad feeling I am going to get deleted :(",
    metadata={"source": "tweet"},
)

documents = [
    document_1,
    document_2,
    document_3,
    document_4,
    document_5,
    document_6,
    document_7,
    document_8,
    document_9,
    document_10,
]
uuids = [str(uuid4()) for _ in range(len(documents))]
uuids

['2f95b73e-4f3e-4b13-9727-37a03476b5ea',
 'a4c95843-2ac0-4553-93c6-8b4ae5c7b4f8',
 '7bebf4de-b4d5-4808-a671-2dea138e5752',
 'cf105eae-0e2f-48b7-b0b0-583df82354ea',
 '38241a48-2e6c-4ac7-807f-96783f236981',
 'db9e5201-b56c-4df9-b523-b74177394c1b',
 'f855de29-eb5b-48c6-a215-1fea14d71f33',
 '2bf1dd87-00cc-454b-94fb-15556b327770',
 '3f86fac1-ef1c-42e9-b3b5-3d98582b20bb',
 'd59f71d8-58c3-4e6b-9960-907b654249d5']

## Similarity search with euclidean distance (Default)

The Kinetica Module will try to create a table with the name of the collection.
So, make sure that the collection name is unique and the user has the permission to create a table.

In [4]:
COLLECTION_NAME = "langchain_example"

vectorstore = KineticaVectorstore(
    config=k_config,
    embedding_function=embeddings,
    collection_name=COLLECTION_NAME,
    pre_delete_collection=True,
)

vectorstore.add_documents(documents=documents, ids=uuids)

print()
print("Similarity Search")
results = vectorstore.similarity_search(
    "LangChain provides abstractions to make working with LLMs easy",
    k=2,
    filter={"source": "tweet"},
)
for res in results:
    print(f"* {res.page_content} [{res.metadata}]")

print()
print("Similarity search with score")
results = vectorstore.similarity_search_with_score(
    "Will it be hot tomorrow?", k=1, emb_filter={"source": "news"}
)
for res, score in results:
    print(f"* [SIM={score:3f}] {res.page_content} [{res.metadata}]")


Similarity Search
* Building an exciting new project with LangChain - come check it out! [{'source': 'tweet'}]
* LangGraph is the best framework for building stateful, agentic applications! [{'source': 'tweet'}]

Similarity search with score
* [SIM=0.945382] The weather forecast for tomorrow is cloudy and overcast, with a high of 62 degrees. [{'source': 'news'}]


## Working with vectorstore

### Adding documents

Above, we created a vectorstore from scratch. However, often times we want to work with an existing vectorstore.
In order to do that, we can initialize it directly.

In [5]:
vectorstore = KineticaVectorstore(
    config=k_config,
    embedding_function=embeddings,
    collection_name=COLLECTION_NAME,
)

# We can add documents to the existing vectorstore.
vectorstore.add_documents([Document(page_content="foo")])

docs_with_score = vectorstore.similarity_search_with_score("foo")

print(f"First result: {docs_with_score[0]}")
print(f"Second result: {docs_with_score[1]}")

First result: (Document(metadata={}, page_content='foo'), 0.0)
Second result: (Document(metadata={'source': 'tweet'}, page_content='Building an exciting new project with LangChain - come check it out!'), 1.2609258890151978)


### Overriding a vectorstore

If you have an existing collection, you override it by doing `from_documents` and setting `pre_delete_collection` = True

In [6]:
vectorstore = KineticaVectorstore.from_documents(
    documents=documents,
    embedding=embeddings,
    collection_name=COLLECTION_NAME,
    config=k_config,
    pre_delete_collection=True,
)

docs_with_score = vectorstore.similarity_search_with_score("foo")
docs_with_score[0]

(Document(metadata={'source': 'tweet'}, page_content='Building an exciting new project with LangChain - come check it out!'),
 1.2609500885009766)

### Using a VectorStore as a retriever

In [7]:
from langchain_core.vectorstores.base import VectorStoreRetriever

retriever: VectorStoreRetriever = vectorstore.as_retriever()
retriever

VectorStoreRetriever(tags=['KineticaVectorstore', 'OpenAIEmbeddings'], vectorstore=<langchain_kinetica.vectorstores.KineticaVectorstore object at 0x12268c050>, search_kwargs={})